In [ ]:
import MODE_utilities as AS

import numpy as np
import pandas as pd
import random
import time
import torch
import matplotlib.pyplot as plt

import MOO_2d_iterate_add_neighbor as moo

In [ ]:
# ============================================================
# LSTM hyperparameter space
# ============================================================

WINDOW_MIN = 5
WINDOW_MAX = 20

LR_MIN = 0.0001
LR_MAX = 0.1

BATCH_MIN = 8
BATCH_MAX = 128

HIDDEN_MIN = 2
HIDDEN_MAX = 10

EPOCH_MIN = 50
EPOCH_MAX = 150

LAYER_MIN = 1
LAYER_MAX = 3


# ============================================================
# MODE settings used for the fairness budget
# ============================================================

MODE_POP_SIZE = 10
MODE_GENERATIONS = 10

# 10 individuals x 10 generations
MAX_EVALUATIONS = 100


# ============================================================
# Cao search settings
# ============================================================

# Same initial number of candidates as MODE's population
INITIAL_SIZE = 10

# This is ONLY an upper bound.
# The real stopping criterion is MAX_EVALUATIONS.
N_ROUNDS = 100

# Preserve Cao et al.'s original per-center setting
PER_CENTER = 7


# ============================================================
# Data settings
# ============================================================

SERIES_LENGTH = 500

# First 80% of every profile used as offline data
OFFLINE_FRACTION = 0.80

# Internal training/validation split during HPO
HPO_SPLIT = 0.70

NORMALIZE = True


# ============================================================
# Final Cao Pareto preference
# ============================================================

CAO_WEIGHTS = np.array([0.6, 0.4])

In [ ]:
SEED = 89

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)
print("Device:", AS.device)

In [ ]:
# ============================================================
# 1. Generate one random LSTM configuration
# ============================================================

def random_candidate():

    return (
        random.randint(WINDOW_MIN, WINDOW_MAX),
        random.uniform(LR_MIN, LR_MAX),
        random.randint(BATCH_MIN, BATCH_MAX),
        random.randint(HIDDEN_MIN, HIDDEN_MAX),
        random.randint(EPOCH_MIN, EPOCH_MAX),
        random.randint(LAYER_MIN, LAYER_MAX),
    )


# ============================================================
# 2. Convert Cao candidate -> LSTM architecture dictionary
# ============================================================

def candidate_to_architecture(candidate):

    window, lr, batch, hidden, epochs, layers = candidate

    return {
        "window": int(window),
        "learning_rate": float(lr),
        "batch_size": int(batch),
        "hidden_dim": int(hidden),
        "epochs": int(epochs),
        "layer_dim": int(layers),
    }


# ============================================================
# 3. Obtain a hashable candidate tuple
#
# Works with either:
#   - a normal candidate tuple
#   - a moo.DataManual object
# ============================================================

def candidate_tuple(x):

    if isinstance(x, moo.DataManual):

        return (
            int(x.window),
            round(float(x.learning_rate), 12),
            int(x.batch_size),
            int(x.hidden_dim),
            int(x.epochs),
            int(x.layer_dim),
        )

    return (
        int(x[0]),
        round(float(x[1]), 12),
        int(x[2]),
        int(x[3]),
        int(x[4]),
        int(x[5]),
    )


# ============================================================
# 4. Train exactly the MODE LSTM on the FULL original data
#
# IMPORTANT:
# "series" is already per-series normalized in Cell 5.
#
# No Soft-DTW.
# No barycenters.
# No K.
# ============================================================

def train_model_from_full_series(
    series,
    architecture,
    split=HPO_SPLIT,
):

    series = np.asarray(series, dtype=np.float64)

    if series.ndim != 2:
        raise ValueError(
            "series must have shape "
            "(n_series, series_length). "
            f"Received {series.shape}."
        )

    series_length = series.shape[1]

    # Concatenate the original normalized profiles.
    Y = series.reshape(-1)

    split_idx = int(split * len(Y))

    Train = Y[:split_idx]
    Val = Y[split_idx:]

    ML = AS.Instance(
        train_data=Train,
        val_data=Val,
        chromosome=architecture,
        NORMALIZED=True,
        series_length=series_length,
        train_offset=0,
        val_offset=split_idx,
    )

    ML.train_phase()

    return ML


# ============================================================
# 5. Evaluate one Cao candidate
#
# Objectives:
#   f1 = validation RMSE
#   f2 = LSTM training time
# ============================================================

def evaluate_candidate(
    candidate,
    data,
    split=HPO_SPLIT,
):

    architecture = candidate_to_architecture(candidate)

    # --------------------------------------------------------
    # LSTM training time
    # --------------------------------------------------------

    AS._sync_device()
    start = time.perf_counter()

    ML_current = train_model_from_full_series(
        series=data,
        architecture=architecture,
        split=split,
    )

    AS._sync_device()

    training_time = time.perf_counter() - start

    # --------------------------------------------------------
    # Validation RMSE
    # --------------------------------------------------------

    val_output = ML_current.validation_phase()

    val_rmse = float(val_output[0])

    # --------------------------------------------------------
    # Move stored model away from GPU.
    #
    # Otherwise every evaluated candidate would remain on the GPU
    # and eventually consume all GPU memory.
    # --------------------------------------------------------

    ML_current.model = ML_current.model.to("cpu")

    return moo.DataManual(
        window=architecture["window"],
        learning_rate=architecture["learning_rate"],
        batch_size=architecture["batch_size"],
        hidden_dim=architecture["hidden_dim"],
        epochs=architecture["epochs"],
        layer_dim=architecture["layer_dim"],
        val_rmse=val_rmse,
        training_time=training_time,
        model_instance=ML_current,
    )


# ============================================================
# 6. Cao-style local neighborhood
#
# One hyperparameter is changed at a time.
# ============================================================

def generate_neighbors(center):

    w, lr, bs, hd, ep, ly = center

    neighbors = set()

    # --------------------------------------------------------
    # Window ±1
    # --------------------------------------------------------

    for delta in (-1, 1):

        new_w = w + delta

        if WINDOW_MIN <= new_w <= WINDOW_MAX:

            neighbors.add(
                (new_w, lr, bs, hd, ep, ly)
            )

    # --------------------------------------------------------
    # Learning rate ×0.5 or ×2
    # --------------------------------------------------------

    for factor in (0.5, 2.0):

        new_lr = lr * factor

        if LR_MIN <= new_lr <= LR_MAX:

            neighbors.add(
                (w, new_lr, bs, hd, ep, ly)
            )

    # --------------------------------------------------------
    # Batch size ±8
    # --------------------------------------------------------

    for delta in (-8, 8):

        new_bs = bs + delta

        if BATCH_MIN <= new_bs <= BATCH_MAX:

            neighbors.add(
                (w, lr, new_bs, hd, ep, ly)
            )

    # --------------------------------------------------------
    # Hidden dimension ±1
    # --------------------------------------------------------

    for delta in (-1, 1):

        new_hd = hd + delta

        if HIDDEN_MIN <= new_hd <= HIDDEN_MAX:

            neighbors.add(
                (w, lr, bs, new_hd, ep, ly)
            )

    # --------------------------------------------------------
    # Epochs ±10
    # --------------------------------------------------------

    for delta in (-10, 10):

        new_ep = ep + delta

        if EPOCH_MIN <= new_ep <= EPOCH_MAX:

            neighbors.add(
                (w, lr, bs, hd, new_ep, ly)
            )

    # --------------------------------------------------------
    # Layers ±1
    # --------------------------------------------------------

    for delta in (-1, 1):

        new_ly = ly + delta

        if LAYER_MIN <= new_ly <= LAYER_MAX:

            neighbors.add(
                (w, lr, bs, hd, ep, new_ly)
            )

    return list(neighbors)

In [ ]:
# ============================================================
# Load CODEF data
# ============================================================

cpu_exp1 = AS.read_experiment_data(1, "CPU")
ram_exp1 = AS.read_experiment_data(1, "RAM")

cpu_exp2 = AS.read_experiment_data(2, "CPU")
ram_exp2 = AS.read_experiment_data(2, "RAM")

cpu_exp3 = AS.read_experiment_data(3, "CPU")
ram_exp3 = AS.read_experiment_data(3, "RAM")

# ============================================================
# Load Raspberru Pi data
# ============================================================

raspberry_RAM = AS.read_raspberry_pi_dataset(series_length=500, column='memory')[:60][:]
raspbery_CPU = AS.read_raspberry_pi_dataset(series_length=500, column='cpu')[:60][:]

# split_data


# offline_data = split_data[:, :int(SPLIT*split_data.shape[1])]
# online_data = split_data[:, int(SPLIT*split_data.shape[1]):]

# ============================================================
# Select experiment/resource
#
# Change only this line when running another experiment.
# ============================================================

My_data = cpu_exp1[:]


# ============================================================
# Form profiles of length SERIES_LENGTH
# ============================================================

num_full_series = len(My_data) // SERIES_LENGTH

trimmed_data = My_data[
    :num_full_series * SERIES_LENGTH
]

split_data = trimmed_data.reshape(
    num_full_series,
    SERIES_LENGTH
)


# ============================================================
# Offline / online split
# ============================================================

offline_length = int(
    OFFLINE_FRACTION * SERIES_LENGTH
)

offline_data = split_data[:, :offline_length]

online_data = split_data[:, offline_length:]


print("Complete profiles:", split_data.shape)
print("Offline data:", offline_data.shape)
print("Online data:", online_data.shape)

In [ ]:
# ============================================================
# Per-series normalization
#
# Same normalization principle used by MODE before Soft-DTW.
# Keep the 2-D structure!
# ============================================================

offline_hpo_data = AS.minmax_normalize_ts(
    offline_data
)

print("Raw offline shape:", offline_data.shape)
print("Normalized HPO shape:", offline_hpo_data.shape)

assert offline_hpo_data.ndim == 2
assert offline_hpo_data.shape == offline_data.shape

assert online_data.shape[1] > WINDOW_MAX

print(
    "Normalized range:",
    offline_hpo_data.min(),
    offline_hpo_data.max()
)

In [ ]:
def run_cao_search(
    data,
    initial_size=INITIAL_SIZE,
    n_rounds=N_ROUNDS,
    per_center=PER_CENTER,
    max_evaluations=MAX_EVALUATIONS,
    seed=SEED,
):

    # ========================================================
    # Reproducibility
    # ========================================================

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # ========================================================
    # Storage
    # ========================================================

    current_sample = []
    sampled_set = set()

    rounds_new_points = []
    fronts_per_round = []

    # ========================================================
    # TOTAL HPO TIMER
    # ========================================================

    AS._sync_device()
    cao_start = time.perf_counter()

    # ========================================================
    # 1. INITIAL SAMPLE
    # ========================================================

    while (
        len(current_sample) < initial_size
        and len(current_sample) < max_evaluations
    ):

        candidate = random_candidate()

        key = candidate_tuple(candidate)

        if key in sampled_set:
            continue

        result = evaluate_candidate(
            candidate=candidate,
            data=data,
            split=HPO_SPLIT,
        )

        current_sample.append(result)
        sampled_set.add(key)

        print(
            f"Initial evaluation "
            f"{len(current_sample)}/{initial_size}: "
            f"RMSE={result.val_rmse:.6f}, "
            f"time={result.training_time:.3f}s"
        )

    # ========================================================
    # Initial Pareto front
    # ========================================================

    initial_pareto = moo.find_pareto_optimal_2d(
        current_sample
    )

    print(
        "\nInitial Pareto front:",
        len(initial_pareto),
        "/",
        len(current_sample),
    )

    # ========================================================
    # 2. ITERATIVE PARETO-NEIGHBORHOOD SEARCH
    # ========================================================

    for r in range(n_rounds):

        # Stop immediately when the common HPO budget is reached.
        if len(current_sample) >= max_evaluations:
            break

        centers = moo.find_pareto_optimal_2d(
            current_sample
        )

        neighbors_this_round = []

        for center_dm in centers:

            if (
                len(current_sample)
                + len(neighbors_this_round)
                >= max_evaluations
            ):
                break

            center = candidate_tuple(
                center_dm
            )

            neighbors = generate_neighbors(
                center
            )

            random.shuffle(neighbors)

            valid_neighbors = []

            for candidate in neighbors:

                key = candidate_tuple(
                    candidate
                )

                if key not in sampled_set:
                    valid_neighbors.append(
                        candidate
                    )

            take = valid_neighbors[
                :per_center
            ]

            for candidate in take:

                # --------------------------------------------
                # HARD EVALUATION-BUDGET STOP
                # --------------------------------------------

                if (
                    len(current_sample)
                    + len(neighbors_this_round)
                    >= max_evaluations
                ):
                    break

                result = evaluate_candidate(
                    candidate=candidate,
                    data=data,
                    split=HPO_SPLIT,
                )

                neighbors_this_round.append(
                    result
                )

                sampled_set.add(
                    candidate_tuple(result)
                )

        # ----------------------------------------------------
        # If this round found nothing new, stop.
        # ----------------------------------------------------

        if len(neighbors_this_round) == 0:

            print(
                "\nNo unseen neighbors remained. "
                "Stopping Cao search."
            )

            break

        # ----------------------------------------------------
        # Add newly trained configurations
        # ----------------------------------------------------

        current_sample.extend(
            neighbors_this_round
        )

        # ----------------------------------------------------
        # Recompute Pareto front
        # ----------------------------------------------------

        front = moo.find_pareto_optimal_2d(
            current_sample
        )

        rounds_new_points.append(
            neighbors_this_round
        )

        fronts_per_round.append(
            front
        )

        print(
            f"\nRound {r + 1}: "
            f"{len(neighbors_this_round)} new evaluations, "
            f"{len(current_sample)} total evaluations, "
            f"{len(front)} Pareto solutions"
        )

        # ----------------------------------------------------
        # Common budget reached
        # ----------------------------------------------------

        if len(current_sample) >= max_evaluations:

            print(
                f"\nEvaluation budget reached: "
                f"{max_evaluations}"
            )

            break

    # ========================================================
    # END TOTAL HPO TIMER
    # ========================================================

    AS._sync_device()

    cao_hpo_time = (
        time.perf_counter()
        - cao_start
    )

    final_pareto = (
        moo.find_pareto_optimal_2d(
            current_sample
        )
    )

    return {
        "current_sample":
            current_sample,

        "initial_pareto":
            initial_pareto,

        "rounds_new_points":
            rounds_new_points,

        "fronts_per_round":
            fronts_per_round,

        "final_pareto":
            final_pareto,

        "hpo_time":
            cao_hpo_time,

        "n_evaluations":
            len(current_sample),
    }

In [ ]:
cao_run = run_cao_search(
    data=offline_hpo_data,
    initial_size=INITIAL_SIZE,
    n_rounds=N_ROUNDS,
    per_center=PER_CENTER,
    max_evaluations=MAX_EVALUATIONS,
)

In [ ]:
current_sample = cao_run["current_sample"]
initial_pareto = cao_run["initial_pareto"]
rounds_new_points = cao_run["rounds_new_points"]
fronts_per_round = cao_run["fronts_per_round"]
final_pareto = cao_run["final_pareto"]

cao_hpo_time = cao_run["hpo_time"]
n_cao_evaluations = cao_run["n_evaluations"]


print("=" * 70)
print("CAO-MOO-LSTM SEARCH SUMMARY")
print("=" * 70)

print("Initial sample:", INITIAL_SIZE)

for r, points in enumerate(
    rounds_new_points,
    start=1,
):
    print(
        f"Round {r}: "
        f"{len(points)} new evaluations"
    )

print(
    "\nTotal evaluations:",
    n_cao_evaluations
)

print(
    "Requested evaluation budget:",
    MAX_EVALUATIONS
)

print(
    "Total HPO time [s]:",
    cao_hpo_time
)

print(
    "Final Pareto solutions:",
    len(final_pareto)
)


if n_cao_evaluations == MAX_EVALUATIONS:
    print("\nEvaluation budgets MATCH.")
else:
    print(
        "\nWARNING: Cao stopped before reaching "
        "the complete evaluation budget."
    )

In [ ]:
rmse = np.array(
    [x.val_rmse for x in final_pareto],
    dtype=float,
)

train_time = np.array(
    [x.training_time for x in final_pareto],
    dtype=float,
)

rmse_n = (
    (rmse - rmse.min())
    /
    (rmse.max() - rmse.min() + 1e-12)
)

time_n = (
    (train_time - train_time.min())
    /
    (train_time.max() - train_time.min() + 1e-12)
)

In [ ]:
score = (
    CAO_WEIGHTS[0] * rmse_n
    +
    CAO_WEIGHTS[1] * time_n
)

best_idx = int(
    np.argmin(score)
)

best_cao = final_pareto[
    best_idx
]

In [ ]:
best_cao_architecture = {
    "window":
        best_cao.window,

    "learning_rate":
        best_cao.learning_rate,

    "batch_size":
        best_cao.batch_size,

    "hidden_dim":
        best_cao.hidden_dim,

    "epochs":
        best_cao.epochs,

    "layer_dim":
        best_cao.layer_dim,
}


print("=" * 70)
print("SELECTED CAO-MOO-LSTM CONFIGURATION")
print("=" * 70)

print(best_cao_architecture)

print(
    "\nValidation RMSE:",
    best_cao.val_rmse
)

print(
    "Selected candidate training time [s]:",
    best_cao.training_time
)

print(
    "Total Cao HPO time [s]:",
    cao_hpo_time
)

print(
    "Total Cao evaluations:",
    n_cao_evaluations
)

In [ ]:
best_cao_instance = (
    best_cao.model_instance
)

best_cao_instance.model = (
    best_cao_instance.model.to(
        AS.device
    )
)

print(
    "Selected model moved to:",
    AS.device
)

In [ ]:
online_results = []

for series_id, ts in enumerate(
    online_data
):

    res = AS.evaluate_one_individual_series(
        ML=best_cao_instance,
        series_raw=ts,
        sdtw_normalize=NORMALIZE,
        return_raw_scale=True,
    )

    online_results.append({
        "series_id":
            series_id,

        "rmse_normalized":
            res["rmse_used"],

        "rmse_raw":
            res["rmse_raw"],

        "nrmse_raw":
            res["nrmse_percent_raw"],

        "predictions":
            res["preds_raw"],

        "targets":
            res["targets_raw"],

        "time_index":
            res["prediction_time_index"],
    })

In [ ]:
online_metrics_df = pd.DataFrame([
    {
        "series_id":
            row["series_id"],

        "rmse_normalized":
            row["rmse_normalized"],

        "rmse_raw":
            row["rmse_raw"],

        "nrmse_raw":
            row["nrmse_raw"],
    }
    for row in online_results
])


print(
    "Mean normalized RMSE:",
    online_metrics_df[
        "rmse_normalized"
    ].mean()
)

print(
    "Std normalized RMSE:",
    online_metrics_df[
        "rmse_normalized"
    ].std()
)

print(
    "Mean raw RMSE:",
    online_metrics_df[
        "rmse_raw"
    ].mean()
)

print(
    "Mean raw NRMSE [%]:",
    online_metrics_df[
        "nrmse_raw"
    ].mean()
)

In [ ]:
cao_summary = {
    "method":
        "Cao-MOO-LSTM",

    "n_evaluations":
        n_cao_evaluations,

    "hpo_time_sec":
        cao_hpo_time,

    "selected_validation_rmse":
        best_cao.val_rmse,

    "selected_training_time_sec":
        best_cao.training_time,

    "online_rmse_normalized_mean":
        online_metrics_df["rmse_normalized"].mean(),

    "online_rmse_normalized_std":
        online_metrics_df["rmse_normalized"].std(),

    "online_rmse_raw_mean":
        online_metrics_df["rmse_raw"].mean(),

    "window":
        best_cao.window,

    "learning_rate":
        best_cao.learning_rate,

    "batch_size":
        best_cao.batch_size,

    "hidden_dim":
        best_cao.hidden_dim,

    "epochs":
        best_cao.epochs,

    "layer_dim":
        best_cao.layer_dim,
}

cao_summary_df = pd.DataFrame(
    [cao_summary]
)

cao_summary_df

In [ ]:
# ============================================================
# Pareto plot
# ============================================================

plt.figure(figsize=(8, 5))

# All evaluated configurations
plt.scatter(
    [
        x.val_rmse
        for x in current_sample
    ],
    [
        x.training_time
        for x in current_sample
    ],
    alpha=0.35,
    label="Evaluated configurations",
)

# Final Pareto front
pareto_sorted = sorted(
    final_pareto,
    key=lambda x: x.val_rmse,
)

plt.plot(
    [
        x.val_rmse
        for x in pareto_sorted
    ],
    [
        x.training_time
        for x in pareto_sorted
    ],
    marker="o",
    linewidth=2,
    label="Final Pareto front",
)

# Selected solution
plt.scatter(
    [best_cao.val_rmse],
    [best_cao.training_time],
    marker="*",
    s=180,
    label="Selected solution",
)

plt.xlabel("Validation RMSE")
plt.ylabel("Training Time (s)")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# Final scalar summary
# ============================================================

cao_summary = {
    "method":
        "Cao-MOO-LSTM",

    "n_evaluations":
        n_cao_evaluations,

    "hpo_time_sec":
        cao_hpo_time,

    "selected_validation_rmse":
        best_cao.val_rmse,

    "selected_training_time_sec":
        best_cao.training_time,

    "online_rmse_normalized_mean":
        online_metrics_df[
            "rmse_normalized"
        ].mean(),

    "online_rmse_normalized_std":
        online_metrics_df[
            "rmse_normalized"
        ].std(),

    "online_rmse_raw_mean":
        online_metrics_df[
            "rmse_raw"
        ].mean(),

    "window":
        best_cao.window,

    "learning_rate":
        best_cao.learning_rate,

    "batch_size":
        best_cao.batch_size,

    "hidden_dim":
        best_cao.hidden_dim,

    "epochs":
        best_cao.epochs,

    "layer_dim":
        best_cao.layer_dim,
}


cao_summary_df = pd.DataFrame(
    [cao_summary]
)

print(cao_summary_df)

In [ ]:
online_results

In [ ]:
with open("Cao_results.txt", "w", encoding="utf-8") as f:

    # HPO information
    f.write("CAO-MOO-LSTM\n")
    f.write("=" * 60 + "\n")
    f.write(f"Total Cao HPO time [s]: {cao_hpo_time:.6f}\n")
    f.write(f"Total evaluations: {n_cao_evaluations}\n")
    f.write(f"Seed: {SEED}\n")

    # Selected configuration
    f.write("\nSelected configuration:\n")
    f.write(f"Window: {best_cao.window}\n")
    f.write(f"Learning rate: {best_cao.learning_rate}\n")
    f.write(f"Batch size: {best_cao.batch_size}\n")
    f.write(f"Hidden dim: {best_cao.hidden_dim}\n")
    f.write(f"Epochs: {best_cao.epochs}\n")
    f.write(f"Layers: {best_cao.layer_dim}\n")

    # RMSE values
    f.write("\n" + "=" * 60 + "\n")
    f.write("RAW-SCALE RMSE FOR EACH TIME SERIES\n")
    f.write("=" * 60 + "\n")

    for row in online_results:
        f.write(
            f"Series {row['series_id']}: "
            f"{row['rmse_raw']:.10f}\n"
        )

    # Summary
    raw_rmse_values = np.array(
        [row["rmse_raw"] for row in online_results],
        dtype=float
    )

    f.write("\n" + "=" * 60 + "\n")
    f.write("RAW-SCALE RMSE SUMMARY\n")
    f.write("=" * 60 + "\n")

    f.write(
        f"Number of time series: "
        f"{len(raw_rmse_values)}\n"
    )

    f.write(
        f"Mean RMSE: "
        f"{np.mean(raw_rmse_values):.10f}\n"
    )

    f.write(
        f"Std RMSE: "
        f"{np.std(raw_rmse_values, ddof=1):.10f}\n"
    )

    f.write(
        f"Min RMSE: "
        f"{np.min(raw_rmse_values):.10f}\n"
    )

    f.write(
        f"Max RMSE: "
        f"{np.max(raw_rmse_values):.10f}\n"
    )

print("Saved all Cao results to Cao_results.txt")

# Loop run

In [ ]:
ram_exp1 = AS.read_experiment_data(1, "RAM")

cpu_exp2 = AS.read_experiment_data(2, "CPU")
ram_exp2 = AS.read_experiment_data(2, "RAM")

cpu_exp3 = AS.read_experiment_data(3, "CPU")
ram_exp3 = AS.read_experiment_data(3, "RAM")

# ============================================================
# Load Raspberru Pi data
# ============================================================

raspberry_RAM = AS.read_raspberry_pi_dataset(series_length=500, column='memory')[:60][:]
raspberry_CPU = AS.read_raspberry_pi_dataset(series_length=500, column='cpu')[:60][:]


In [ ]:
def prepare_experiment_data(
    data,
    series_length=SERIES_LENGTH,
    offline_fraction=OFFLINE_FRACTION,
):
    """
    Convert either:
        1-D flat CODEF data
    or:
        2-D Raspberry Pi data

    into:
        split_data   : complete time series
        offline_data : first offline_fraction of each series
        online_data  : remaining part
    """

    data = np.asarray(data, dtype=np.float64)

    # ============================================================
    # Case 1: flat CODEF array
    # ============================================================

    if data.ndim == 1:

        num_full_series = len(data) // series_length

        trimmed_data = data[
            :num_full_series * series_length
        ]

        split_data = trimmed_data.reshape(
            num_full_series,
            series_length,
        )

    # ============================================================
    # Case 2: already separated time series
    # e.g. Raspberry Pi: (60, 500)
    # ============================================================

    elif data.ndim == 2:

        if data.shape[1] != series_length:
            raise ValueError(
                f"Expected series length {series_length}, "
                f"but received shape {data.shape}."
            )

        split_data = data.copy()

    else:

        raise ValueError(
            f"Expected 1-D or 2-D data, got shape {data.shape}."
        )

    # ============================================================
    # Offline / online split
    # ============================================================

    offline_length = int(
        offline_fraction * series_length
    )

    offline_data = split_data[
        :, :offline_length
    ]

    online_data = split_data[
        :, offline_length:
    ]

    return (
        split_data,
        offline_data,
        online_data,
    )

In [ ]:
def run_and_save_cao_experiment(
    experiment_name,
    data,
    output_folder="CAO_results",
):

    import os

    os.makedirs(
        output_folder,
        exist_ok=True,
    )

    print("\n" + "=" * 80)
    print(f"STARTING: {experiment_name}")
    print("=" * 80)

    # ============================================================
    # 1. Prepare data
    # ============================================================

    (
        split_data,
        offline_data,
        online_data,
    ) = prepare_experiment_data(
        data=data,
        series_length=SERIES_LENGTH,
        offline_fraction=OFFLINE_FRACTION,
    )

    print("Complete data shape:", split_data.shape)
    print("Offline shape:", offline_data.shape)
    print("Online shape:", online_data.shape)

    # ============================================================
    # 2. Normalize OFFLINE data for HPO
    # ============================================================

    offline_hpo_data = (
        AS.minmax_normalize_ts(
            offline_data
        )
    )

    print(
        "Normalized offline range:",
        offline_hpo_data.min(),
        offline_hpo_data.max(),
    )

    # ============================================================
    # 3. Run Cao MOO search
    # ============================================================

    cao_run = run_cao_search(
        data=offline_hpo_data,
        initial_size=INITIAL_SIZE,
        n_rounds=N_ROUNDS,
        per_center=PER_CENTER,
        max_evaluations=MAX_EVALUATIONS,
    )

    current_sample = (
        cao_run["current_sample"]
    )

    final_pareto = (
        cao_run["final_pareto"]
    )

    cao_hpo_time = (
        cao_run["hpo_time"]
    )

    n_cao_evaluations = (
        cao_run["n_evaluations"]
    )

    # ============================================================
    # 4. Normalize final Pareto objectives
    # ============================================================

    rmse = np.array(
        [
            x.val_rmse
            for x in final_pareto
        ],
        dtype=float,
    )

    train_time = np.array(
        [
            x.training_time
            for x in final_pareto
        ],
        dtype=float,
    )

    rmse_n = (
        (rmse - rmse.min())
        /
        (
            rmse.max()
            - rmse.min()
            + 1e-12
        )
    )

    time_n = (
        (train_time - train_time.min())
        /
        (
            train_time.max()
            - train_time.min()
            + 1e-12
        )
    )

    # ============================================================
    # 5. Select final Cao solution
    # ============================================================

    score = (
        CAO_WEIGHTS[0] * rmse_n
        +
        CAO_WEIGHTS[1] * time_n
    )

    best_idx = int(
        np.argmin(score)
    )

    best_cao = final_pareto[
        best_idx
    ]

    best_cao_instance = (
        best_cao.model_instance
    )

    # Put only selected model back on execution device
    best_cao_instance.model = (
        best_cao_instance.model.to(
            AS.device
        )
    )

    # ============================================================
    # 6. Evaluate ALL online time series
    # ============================================================

    online_results = []

    for series_id, ts in enumerate(
        online_data
    ):

        res = (
            AS.evaluate_one_individual_series(
                ML=best_cao_instance,
                series_raw=ts,
                sdtw_normalize=NORMALIZE,
                return_raw_scale=True,
            )
        )

        online_results.append({
            "series_id":
                int(series_id),

            "rmse_normalized":
                float(res["rmse_used"]),

            "rmse_raw":
                float(res["rmse_raw"]),

            "nrmse_raw":
                float(
                    res[
                        "nrmse_percent_raw"
                    ]
                ),
        })

    # ============================================================
    # 7. Raw RMSE array
    # ============================================================

    raw_rmse_values = np.array(
        [
            row["rmse_raw"]
            for row in online_results
        ],
        dtype=float,
    )

    normalized_rmse_values = np.array(
        [
            row["rmse_normalized"]
            for row in online_results
        ],
        dtype=float,
    )

    # ============================================================
    # 8. Save TXT
    # ============================================================

    txt_path = os.path.join(
        output_folder,
        f"Cao_{experiment_name}_results.txt",
    )

    with open(
        txt_path,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            f"CAO-MOO-LSTM - {experiment_name}\n"
        )

        f.write("=" * 70 + "\n")

        # --------------------------------------------------------
        # Experiment setup
        # --------------------------------------------------------

        f.write("\nEXPERIMENT SETUP\n")
        f.write("-" * 70 + "\n")

        f.write(
            f"Number of time series: "
            f"{len(split_data)}\n"
        )

        f.write(
            f"Original series length: "
            f"{SERIES_LENGTH}\n"
        )

        f.write(
            f"Offline length: "
            f"{offline_data.shape[1]}\n"
        )

        f.write(
            f"Online length: "
            f"{online_data.shape[1]}\n"
        )

        f.write(
            f"Initial size: "
            f"{INITIAL_SIZE}\n"
        )

        f.write(
            f"Maximum evaluations: "
            f"{MAX_EVALUATIONS}\n"
        )

        f.write(
            f"Actual evaluations: "
            f"{n_cao_evaluations}\n"
        )

        f.write(
            f"PER_CENTER: "
            f"{PER_CENTER}\n"
        )

        f.write(
            f"Seed: "
            f"{SEED}\n"
        )

        # --------------------------------------------------------
        # HPO time
        # --------------------------------------------------------

        f.write("\nHPO RESULTS\n")
        f.write("-" * 70 + "\n")

        f.write(
            f"Total Cao HPO time [s]: "
            f"{cao_hpo_time:.10f}\n"
        )

        f.write(
            f"Final Pareto solutions: "
            f"{len(final_pareto)}\n"
        )

        # --------------------------------------------------------
        # Selected architecture
        # --------------------------------------------------------

        f.write(
            "\nSELECTED CONFIGURATION\n"
        )

        f.write("-" * 70 + "\n")

        f.write(
            f"Window: "
            f"{best_cao.window}\n"
        )

        f.write(
            f"Learning rate: "
            f"{best_cao.learning_rate}\n"
        )

        f.write(
            f"Batch size: "
            f"{best_cao.batch_size}\n"
        )

        f.write(
            f"Hidden dim: "
            f"{best_cao.hidden_dim}\n"
        )

        f.write(
            f"Epochs: "
            f"{best_cao.epochs}\n"
        )

        f.write(
            f"Layers: "
            f"{best_cao.layer_dim}\n"
        )

        f.write(
            f"Validation RMSE: "
            f"{best_cao.val_rmse:.10f}\n"
        )

        f.write(
            f"Selected candidate "
            f"training time [s]: "
            f"{best_cao.training_time:.10f}\n"
        )

        # --------------------------------------------------------
        # All raw-scale RMSE values
        # --------------------------------------------------------

        f.write(
            "\nRAW-SCALE RMSE FOR EACH "
            "ONLINE TIME SERIES\n"
        )

        f.write("=" * 70 + "\n")

        for row in online_results:

            f.write(
                f"Series "
                f"{row['series_id']}: "
                f"{row['rmse_raw']:.10f}\n"
            )

        # --------------------------------------------------------
        # Raw RMSE summary
        # --------------------------------------------------------

        f.write(
            "\nRAW-SCALE RMSE SUMMARY\n"
        )

        f.write("=" * 70 + "\n")

        f.write(
            f"Number of time series: "
            f"{len(raw_rmse_values)}\n"
        )

        f.write(
            f"Mean RMSE: "
            f"{np.mean(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Std RMSE: "
            f"{np.std(raw_rmse_values, ddof=1):.10f}\n"
        )

        f.write(
            f"Median RMSE: "
            f"{np.median(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Min RMSE: "
            f"{np.min(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Max RMSE: "
            f"{np.max(raw_rmse_values):.10f}\n"
        )

        # --------------------------------------------------------
        # Normalized RMSE summary
        # --------------------------------------------------------

        f.write(
            "\nNORMALIZED RMSE SUMMARY\n"
        )

        f.write("=" * 70 + "\n")

        f.write(
            f"Mean normalized RMSE: "
            f"{np.mean(normalized_rmse_values):.10f}\n"
        )

        f.write(
            f"Std normalized RMSE: "
            f"{np.std(normalized_rmse_values, ddof=1):.10f}\n"
        )

    print("\nCompleted:", experiment_name)
    print(
        "Total HPO time [s]:",
        cao_hpo_time,
    )
    print(
        "Mean raw RMSE:",
        np.mean(raw_rmse_values),
    )
    print(
        "Saved:",
        txt_path,
    )

    # ============================================================
    # 9. Free selected model/GPU memory
    # ============================================================

    best_cao_instance.model = (
        best_cao_instance.model.to("cpu")
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ============================================================
    # 10. Return useful results
    # ============================================================

    return {
        "experiment":
            experiment_name,

        "hpo_time":
            cao_hpo_time,

        "n_evaluations":
            n_cao_evaluations,

        "best_cao":
            best_cao,

        "final_pareto":
            final_pareto,

        "raw_rmse_values":
            raw_rmse_values,

        "normalized_rmse_values":
            normalized_rmse_values,

        "online_results":
            online_results,

        "txt_path":
            txt_path,
    }

In [ ]:
remaining_experiments = {
    "RAM_EXP1": ram_exp1,
    "CPU_EXP2": cpu_exp2,
    "RAM_EXP2": ram_exp2,
    "CPU_EXP3": cpu_exp3,
    "RAM_EXP3": ram_exp3,
    "RASPBERRY_RAM": raspberry_RAM,
    "RASPBERRY_CPU": raspberry_CPU,
}

In [ ]:
all_cao_results = {}

for experiment_name, experiment_data in remaining_experiments.items():

    result = run_and_save_cao_experiment(
        experiment_name=experiment_name,
        data=experiment_data,
        output_folder="CAO_results",
    )

    all_cao_results[
        experiment_name
    ] = result